In [11]:
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import json
import os
import pandas as pd


folder = 'result_example/natsr_ETTh1_pl24_olfull_optsgd_tb1_2025_09_22_17_46_75e6'
feature = 0
trial = 0

mae_cumul = np.load(folder + '/mae.npy')
mse_cumul = np.load(folder + '/mse.npy')
metrics = np.load(folder + '/metrics.npy')
preds = np.load(folder + '/preds.npy')
trues = np.load(folder + '/trues.npy')
with open(folder + '/args.json') as f:
    args = json.load(f)

print('lr: ', args['online_lr'], 'regul ', args['NatSR_regul'], 'regul_last', args['NatSR_regul_last'])

print('MAE: ', metrics[trial, 0], ' MSE: ', metrics[trial, 1]) 
print(metrics)


fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(len(mae_cumul[trial])), y=mae_cumul[trial], mode='lines', name='MAE'))
fig.add_trace(go.Scatter(x=np.arange(len(mse_cumul[trial])), y=mse_cumul[trial], mode='lines', name='MSE'))
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(len(preds[trial, :, feature])), y=preds[trial, :, feature], mode='lines', name='Predictions'))
fig.add_trace(go.Scatter(x=np.arange(len(trues[trial, :, feature])), y=trues[trial, :, feature], mode='lines', name='True'))
fig.show()

max = 0
feat = 0
for i in range(preds.shape[2]):
    mse = np.mean((preds[trial, :, i] - trues[trial, :, i]) ** 2)
    if mse > max:
        max = mse
        feat = i
print('Max MSE: ', max, ' for feature ', feat)
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(len(preds[trial, :, feat])), y=preds[trial, :, feat], mode='lines', name='Predictions'))
fig.add_trace(go.Scatter(x=np.arange(len(trues[trial, :, feat])), y=trues[trial, :, feat], mode='lines', name='True'))
fig.show()


lr:  0.0954816751881752 regul  0.5 regul_last 0.5
MAE:  0.3507492491605386  MSE:  0.2619659262419389
[[3.50749249e-01 2.61965926e-01 4.93030858e-01 1.59856512e+00
  2.08589125e+02 9.46262014e+02]]


Max MSE:  1.2431074  for feature  163


In [12]:
import os
import json
import numpy as np
import pandas as pd
import datetime
# Base results directory
BASE_DIR = 'result_example/natsr_ETTh1_pl24_olfull_optsgd_tb1_2025_09_22_17_46_75e6'
i = 5

# %% [markdown]
# ## 2. Collect all records

# %%
records = []

# iterate subdirectories
for root, dirs, files in os.walk(BASE_DIR):
    if 'args.json' in files and 'metrics.npy' in files:
        # 1. parse the folder basename
        folder = os.path.basename(root)
        parts = folder.split('_')
        # assume last 6 parts are [YYYY,MM,DD,HH,MM,<run_id>]
        date_str = '_'.join(parts[-6:-1])
        timestamp = datetime.datetime.strptime(date_str, '%Y_%m_%d_%H_%M')
        
        # 2. load args & metrics
        with open(os.path.join(root, 'args.json'), 'r') as f:
            args = json.load(f)
        metrics = np.load(os.path.join(root, 'metrics.npy'))
        mae = metrics[:i, 0].mean()
        mae_sd = metrics[:i, 0].std()
        exp_time = metrics[:i, 5].mean()

        if len(metrics[:i, 0]) != len(set(metrics[:i, 0])):
            print(f"Warning: Duplicate MAE values found in {root}.")

        #Load predictions and true values 
        preds = np.load(os.path.join(root, 'preds.npy'))
        trues = np.load(os.path.join(root, 'trues.npy'))   

        #Compute the MAE of the naive predictor (predicting the last observed value)
        naive_mae = np.mean(np.abs(trues[:i] - np.roll(trues[:i], shift=1, axis=1)))
        mase = metrics[:i, 0] / naive_mae
        mase_sd = mase.std()
        mase = mase.mean()
        mse = metrics[:i,1].mean()

        
        # 3. build record, including the parsed timestamp
        rec = {
            'method':          args.get('method'),
            'data':            args.get('data'),
            'online_lr':    args.get('online_lr'),
            'deg_f':           args.get('deg_f'),
            'MASE':             mase,
            'MASE_sd':             mase_sd,
            'run_timestamp':   timestamp,              # new
            'run_date':        timestamp.date(),        # if you only care about the date
            'exp_time':        exp_time,
            'NatSR_regul':    args.get('NatSR_regul'),
            'NatSR_alpha_ema':    args.get('NatSR_alpha_ema'),
            'pred_len': args.get('pred_len'),
            'NatSR_alpha_ema_grad': args.get('NatSR_alpha_ema_grad'),
            'MSE': mse,
            'pred_len': args.get('pred_len'),
        }
        records.append(rec)
# 4. DataFrame
df = pd.DataFrame.from_records(records)

# 5. Pivot including the date
table = df.pivot_table(
    index=['method', 'online_lr', 'deg_f', 'pred_len', 'NatSR_alpha_ema_grad'],
    columns='data',
    values=['MASE', 'MASE_sd', 'exp_time', 'MSE'],
).sort_index()

display(table)


,,,,,MASE,MASE_sd,MSE,exp_time
,,,,data,ETTh1,ETTh1,ETTh1,ETTh1
method,online_lr,deg_f,pred_len,NatSR_alpha_ema_grad,,,,
ocar,0.095482,50,24,0.9,0.987031,0.0,0.261966,946.262014
